# BER on a Kaggle TPU v5e-8 VM (CPU pipeline, target < 3 h)

The TPU chips cannot run this pipeline: blocking (FAISS), string features and gradient boosting
are CPU work, and the optional transformer cross-encoder needs CUDA. The TPU **VM**, however, has
many more CPU cores and much more RAM than the GPU box, and the pipeline is CPU-bound. This notebook
uses them:

* all cores (capped by `N_JOBS`) for normalisation, TF-IDF, FAISS, rapidfuzz and LightGBM;
* 1M-record chunks (large RAM);
* training on **50 % of states** (whole states, so look-alike density is unchanged; on real data
  this scored 0.9819 nested CV) and 80k training entities;
* stage 2 and the cross-encoder **off**: stage 2 added only +0.0004 but costs about an hour on CPU.

Settings: **Accelerator = TPU VM v5e-8**, **Internet = On**, **Persistence = Files**.

In [ ]:
import os, time, multiprocessing
T0 = time.time()
!nproc; free -g | head -2; df -h /kaggle/working | tail -1
N_JOBS = min(os.cpu_count(), 32)        # forked workers / threads; raise to 48 if RAM is plentiful
print("using", N_JOBS, "workers")

In [ ]:
REPO = "https://github.com/MounishSenisetty/BER.git"
BRANCH = "claude/cool-newton-qy2lr2"      # switch to "main" once PR #7 is merged
CODE = "/kaggle/working/BER"
if not os.path.exists(CODE):
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 -b $BRANCH $REPO $CODE
!cd $CODE && git log --oneline -1
!pip install -q rapidfuzz jellyfish faiss-cpu lightgbm scikit-learn pandas scipy joblib 2>&1 | tail -1
import faiss, lightgbm
print("faiss", faiss.__version__, "| lightgbm", lightgbm.__version__)

In [ ]:
import glob, subprocess
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
if hits:
    DATA = os.path.dirname(os.path.dirname(hits[0]))
else:
    subprocess.run(f"cd {CODE} && git lfs install --local && git lfs pull", shell=True, check=True)
    DATA = f"{CODE}/student_resource/dataset"
TRAIN, TEST = f"{DATA}/train", f"{DATA}/test"
ROOT, WORK = os.path.dirname(DATA), "/kaggle/working"
os.environ.update(TRAIN=TRAIN, TEST=TEST, ROOT=ROOT, WORK=WORK, CODE=CODE, BER_MAX_THREADS=str(N_JOBS),
                  OMP_NUM_THREADS=str(N_JOBS))
print(TRAIN, TEST, sep="\n")

## Train (~40-70 min, estimated)
Watch the first `[us] records 0-1000000 ... done in Ns` line. The US has ~3 of these chunks and India ~2
at 50 % of states. If one chunk takes much more than ~6 min, stop and lower `STATE_FRAC`.

In [ ]:
%%time
STATE_FRAC, TRAIN_ENTITIES = 0.5, 80000
if not os.path.exists(f"{WORK}/models/model_bundle.pkl"):
    !cd $CODE && python -u -m src.train --data-dir "$TRAIN" --model-dir $WORK/models --cache-dir /tmp/ber_s2 --backend lightgbm --no-stage2 --cross-encoder off --train-state-frac $STATE_FRAC --train-entities $TRAIN_ENTITIES --chunk-records 1000000 --n-jobs $N_JOBS 2>&1 | tee $WORK/train.log | grep -vE "prepare \+ block \.\.\.$|features for .* \.\.\.$"
else:
    print("models/model_bundle.pkl exists - skipping training")
print(f"elapsed so far: {(time.time() - T0) / 60:.0f} min")

In [ ]:
import json
r = json.load(open(f"{WORK}/models/cv_report.json"))
print("blocking:", {k: v for k, v in r["blocking"].items() if k != "recall_at_cap"})
print("variants (nested-CV macro F0.5):", r.get("variants"), "-> final:", r.get("final"))
print("S1 dropout:", r.get("s1_drop"), "| lexicon entries:", r.get("lexicon_size"))
!grep -E "State subset|Training pairs|Memory" $WORK/train.log
!sed -n '/Loss report/,/Examples FP/p' $WORK/train.log | head -40

## Inference on the full test split (~60-90 min, estimated)
The test has ~10M records, which is ~10 chunks of 1M. Multiply the time of the first chunk by 10 to
see where you will land.

In [ ]:
%%time
!cd $CODE && python -u -m src.inference --data-dir "$TEST" --model-dir $WORK/models --out-dir $WORK/output --chunk-records 1000000 --n-jobs $N_JOBS 2>&1 | tee $WORK/inference.log | grep -vE "prepare \+ block \.\.\.$|scoring for .* \.\.\.$"
print(f"elapsed so far: {(time.time() - T0) / 60:.0f} min")

In [ ]:
!python "$ROOT/utils/validate_submission.py" --matching $WORK/output/matching_results.tsv --candidate $WORK/output/candidate_pairs.tsv --test-dir "$TEST"
!cd $CODE && python scripts/make_submission_zip.py --team YOUR_TEAM --out-dir $WORK/output --doc Documentation_template.md --dest $WORK
!ls -la $WORK/*.zip
print(f"total: {(time.time() - T0) / 60:.0f} min")